<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 18 - Práctica Integradora: Dashboard Completo

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** Proyecto integrador - Dashboard de supermercados

---

## Objetivos
- Integrar todos los conocimientos de RA2
- Crear un cuadro de mando completo y funcional
- Aplicar metodología CRISP-DM

## 1. Generación del dataset

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# Generar datos realistas de supermercado
n_ventas = 10000
fechas = pd.date_range('2024-01-01', '2024-12-31', freq='D')

tiendas = ['Centro', 'Norte', 'Sur', 'Este', 'Oeste']
categorias = ['Alimentación', 'Bebidas', 'Limpieza', 'Hogar', 'Frescos']

ventas = pd.DataFrame({
    'fecha': np.random.choice(fechas, n_ventas),
    'tienda': np.random.choice(tiendas, n_ventas),
    'categoria': np.random.choice(categorias, n_ventas),
    'unidades': np.random.randint(1, 50, n_ventas),
    'precio_unitario': np.round(np.random.uniform(1, 100, n_ventas), 2),
    'cliente_id': np.random.randint(1, 500, n_ventas)
})

ventas['total'] = ventas['unidades'] * ventas['precio_unitario']
ventas['mes'] = ventas['fecha'].dt.to_period('M').astype(str)
ventas['dia_semana'] = ventas['fecha'].dt.day_name()

print(f"Dataset generado: {len(ventas)} registros")
print(ventas.head())

## 2. Dashboard completo con Plotly

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Calcular KPIs
ventas_total = ventas['total'].sum()
num_transacciones = len(ventas)
ticket_medio = ventas_total / num_transacciones
clientes_unicos = ventas['cliente_id'].nunique()
unidades_vendidas = ventas['unidades'].sum()

# Objetivos
objetivo_ventas = 15000000
objetivo_ticket = 150

# Crear dashboard
fig = make_subplots(
    rows=4, cols=3,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
        [{"type": "scatter", "colspan": 2}, None, {"type": "bar"}],
        [{"type": "pie"}, {"type": "bar", "colspan": 2}, None]
    ],
    subplot_titles=('', '', '', '', '', '',
                    'Evolución de Ventas', 'Ventas por Tienda',
                    'Por Categoría', 'Ventas por Día de Semana'),
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

# Fila 1: KPIs principales
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=ventas_total,
    number={'prefix': '€', 'valueformat': ',.0f'},
    delta={'reference': objetivo_ventas, 'relative': True},
    title={'text': 'Ventas Totales'}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=num_transacciones,
    number={'valueformat': ','},
    title={'text': 'Transacciones'}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=ticket_medio,
    number={'prefix': '€', 'valueformat': '.2f'},
    delta={'reference': objetivo_ticket},
    title={'text': 'Ticket Medio'}
), row=1, col=3)

# Fila 2: KPIs secundarios
fig.add_trace(go.Indicator(
    mode="number",
    value=clientes_unicos,
    number={'valueformat': ','},
    title={'text': 'Clientes Únicos'}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number",
    value=unidades_vendidas,
    number={'valueformat': ','},
    title={'text': 'Unidades Vendidas'}
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=ventas_total/objetivo_ventas*100,
    number={'suffix': '%'},
    title={'text': '% Objetivo'},
    gauge={'axis': {'range': [0, 120]},
           'bar': {'color': 'darkblue'},
           'threshold': {'value': 100, 'thickness': 0.75}}
), row=2, col=3)

# Fila 3: Tendencia y tiendas
ventas_mes = ventas.groupby('mes')['total'].sum().reset_index()
fig.add_trace(go.Scatter(
    x=ventas_mes['mes'], y=ventas_mes['total'],
    mode='lines+markers', name='Ventas',
    line=dict(color='#1f77b4', width=3)
), row=3, col=1)

ventas_tienda = ventas.groupby('tienda')['total'].sum().sort_values(ascending=True).reset_index()
fig.add_trace(go.Bar(
    x=ventas_tienda['total'], y=ventas_tienda['tienda'],
    orientation='h', marker_color='#2ca02c'
), row=3, col=3)

# Fila 4: Categorías y días
ventas_cat = ventas.groupby('categoria')['total'].sum()
fig.add_trace(go.Pie(
    labels=ventas_cat.index, values=ventas_cat.values,
    hole=0.4
), row=4, col=1)

orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ventas_dia = ventas.groupby('dia_semana')['total'].sum().reindex(orden_dias).reset_index()
fig.add_trace(go.Bar(
    x=ventas_dia['dia_semana'], y=ventas_dia['total'],
    marker_color='#ff7f0e'
), row=4, col=2)

fig.update_layout(
    title={'text': '📊 DASHBOARD SUPERMERCADOS 2024', 'x': 0.5, 'font': {'size': 24}},
    height=1000,
    showlegend=False,
    template='plotly_white'
)

fig.show()

## 3. Documentación CRISP-DM

### 1. Business Understanding
- **Objetivo:** Monitorizar ventas de cadena de supermercados
- **KPIs principales:** Ventas, Ticket medio, Transacciones

### 2. Data Understanding
- **Fuentes:** Sistema de TPV
- **Variables:** fecha, tienda, categoría, unidades, precio

### 3. Data Preparation
- Cálculo de totales
- Extracción de mes y día de semana
- Agregaciones por dimensiones

### 4. Modeling
- Dashboard interactivo con Plotly
- 6 KPIs con contexto

### 5. Evaluation
- Cumple requisitos de visualización
- Información accionable

### 6. Deployment
- Exportable como HTML
- Actualizable con nuevos datos

In [ ]:
# Exportar dashboard a HTML
fig.write_html('dashboard_supermercado.html')
print("Dashboard exportado a 'dashboard_supermercado.html'")

---

## Checklist de entrega

- [ ] Dashboard con 4-6 KPIs
- [ ] Visualizaciones de tendencia
- [ ] Análisis por dimensiones
- [ ] Documentación CRISP-DM
- [ ] Código documentado